# Lab 2 — Tools, agents-as-tools, and handoffs

We'll build an **automated sales outreach (SDR) system** for a fictional company, **FlowOps** (an AI workflow-automation SaaS).

You'll learn three new ideas:
1. **Tools** — give an agent a Python function it can call.
2. **Agents as tools** — let one agent use other agents as tools.
3. **Handoffs** — let one agent pass control to another.

> To keep setup simple, "sending" an email is **mocked** (we just print it). In production you'd swap in a real email provider.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
import asyncio

load_dotenv(override=True)

## Step 1 — Three sales agents with different styles

Same job, three personalities. The only thing that changes is the **instructions** (system prompt).

In [ ]:
instructions1 = "You are a professional, serious sales agent for FlowOps, an AI workflow-automation SaaS. You write formal, concise cold emails."
instructions2 = "You are a witty, engaging sales agent for FlowOps, an AI workflow-automation SaaS. You write emails likely to get a reply."
instructions3 = "You are a busy, to-the-point sales agent for FlowOps, an AI workflow-automation SaaS. You write very short cold emails."

sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-5.4-mini")
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-5.4-mini")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-5.4-mini")

In [ ]:
# Run all three in parallel and read their drafts
message = "Write a cold sales email to a prospective customer."

with trace("Three cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

for r in results:
    print(r.final_output, "\n---\n")

## Step 2 — A tool

A **tool** is a normal Python function with `@function_tool` on top. The agent decides when to call it — we don't call it ourselves. No JSON boilerplate needed.

In [ ]:
@function_tool
def send_email(body: str):
    """Send an email with the given body to a sales prospect."""
    # MOCK: in production this would call a real email provider.
    print("----- (MOCK) EMAIL SENT -----")
    print(body)
    print("-----------------------------")
    return {"status": "success"}

send_email  # the decorator turned it into a tool the agent can call

## Step 3 — Agents as tools

We can turn each sales agent into a **tool** with `.as_tool()`. Now a single "manager" agent can call them.

In [ ]:
description = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]
tools

In [ ]:
# The Sales Manager: an agent whose tools are other agents + the send tool.
manager_instructions = """
You are a Sales Manager at FlowOps. Use the three sales_agent tools to generate three cold email drafts.
Review them, pick the single best one, and send ONLY that one using the send_email tool.
You must use the sales_agent tools to write the drafts — do not write them yourself.
Send exactly one email.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=tools,
    model="gpt-5.4-mini",
)

with trace("Sales manager"):
    result = await Runner.run(sales_manager, "Send a cold sales email addressed to 'Dear CEO'.")
print(result.final_output)

The manager **used the three writer agents as tools**, judged the drafts, and called `send_email`. With **tools, control comes back** to the manager after each call.

## Step 4 — Handoffs

A **handoff** is different: the agent **passes control across** to another agent, which finishes the job. Let's give the work of formatting + sending to a dedicated **Email Manager**, and have the Sales Manager hand off to it.

In [ ]:
subject_writer = Agent(name="Subject Writer", instructions="Write a compelling subject line for the given cold email.", model="gpt-5.4-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject line for a cold email")

@function_tool
def send_email_with_subject(subject: str, body: str):
    """Send an email with the given subject and body."""
    print("----- (MOCK) EMAIL SENT -----")
    print("Subject:", subject)
    print(body)
    print("-----------------------------")
    return {"status": "success"}

emailer_agent = Agent(
    name="Email Manager",
    instructions="You format and send emails. First use subject_writer to create a subject, then use send_email_with_subject to send it.",
    tools=[subject_tool, send_email_with_subject],
    model="gpt-5.4-mini",
    handoff_description="Write a subject line and send the email",
)

In [ ]:
# The Sales Manager now has 3 writer tools AND can hand off to the Email Manager.
manager_instructions = """
You are a Sales Manager at FlowOps. Use the three sales_agent tools to generate three drafts and pick the best one.
Then hand off ONLY the best draft to the Email Manager, who will add a subject and send it.
Do not write the drafts yourself. Hand off exactly one email.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=[tool1, tool2, tool3],
    handoffs=[emailer_agent],
    model="gpt-5.4-mini",
)

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, "Send a cold sales email addressed to 'Dear CEO'.")
print(result.final_output)

## Tools vs handoffs — the key difference

- **Tool:** the agent calls it and **control returns** to the agent.
- **Handoff:** the agent **passes control across** to another agent, which takes over.

Check the **Automated SDR** trace at https://platform.openai.com/traces — you'll see the manager call the writer tools, then hand off to the Email Manager.

## Recap
- `@function_tool` turns a function into a tool.
- `.as_tool()` turns an agent into a tool.
- `handoffs=[...]` lets an agent pass control to another agent.

Next, **Lab 3**: making outputs **structured** and adding **guardrails**.